# Transcriptional time per embryo

Infers a transcriptional time per embryo, separately for KO and host cells, by matching cells to WT atlas cells on a joint kNN graph. No figure here.

Cheng, Stok Ranen et al., bioRxiv, doi:10.64898/2026.05.14.725081.

- **ED 2d**: transcriptional age vs dissection day, plotted in `ExtData_RNA.ipynb` from this notebook's table

Inputs:
- `my_data/full_dataset.h5ad` and scdb matrix `full_dataset` (query cells)
- WT atlas cells and metacells under `spatwt_dir`
- misha TSS intervals
- `my_data/ExE/ExE_gene_names.csv`

Output: `my_data/ref/transcriptional_time_best_ref.csv` (time per embryo and KO/host fraction).

Caveats:
- Runs with `dnmt_final_dir` as working directory (needs its `scdb/` and `my_data/`) and the older `metacell` R package.
- Never run in the stored notebook. Moved from `DNMT_final/RNA.5_time_embryos.ipynb`.
- `out_path()` writes the output to `git_test/external/<full path>`; `ExtData_RNA.ipynb` reads the original in `dnmt_final_dir/my_data/ref/`, or this copy with `use_regenerated_inputs = TRUE` (`paths.R`).
- Intermediate scdb objects (`full_dataset_f`, `ts_atlas`, `query_atlas`, `query_atlas_f`) are written straight to `dnmt_final_dir/scdb/`.

## Initialization

In [ ]:
source("../paths.R")
source(file.path(repo_dir, "scripts/transcriptional_time_func.r"))
setwd(dnmt_final_dir)

In [ ]:
library(metacell)
library(anndata)
library(tibble)
library(dplyr)
library(tidyr)
library(tgstat)
library(Matrix)
library(RColorBrewer)
library(ggplot2)
library(ggpubr)

In [ ]:
mat_nm = "full_dataset_f"
atlas_nm = "ts_atlas"
merged_nm = "query_atlas"
merged_nm_f = "query_atlas_f"
scdb_init("./scdb", force_reinit = T)

## Query cells

Restricts scdb matrix `full_dataset` to the cells in `my_data/full_dataset.h5ad`; saved as `full_dataset_f`.

In [ ]:
cells = read_h5ad("./my_data/full_dataset.h5ad")

In [ ]:
query_mat = scdb_mat("full_dataset")

In [ ]:
nonmodel_cells = unname(colnames(query_mat@mat)[!colnames(query_mat@mat) %in% rownames(cells$obs)])
query_mat_f = scm_ignore_cells(scmat = query_mat, ig_cells = nonmodel_cells, reverse = FALSE)

In [ ]:
scdb_add_mat(mat_nm, query_mat_f)

In [ ]:
ref_mcs_path = paste0(spatwt_dir, "/spatwt.metacells.h5ad")
ref_mcs = read_h5ad(ref_mcs_path)

ref_cells_path = paste0(spatwt_dir, "/spatwt.cells.h5ad")
ref_cells = read_h5ad(ref_cells_path)

## Joint query + atlas matrix

### Atlas matrix

Saves the atlas UMI matrix as scdb matrix `ts_atlas`.

In [ ]:
ref_cells_mat = t(ref_cells$X)
atlas = scm_new_matrix(ref_cells_mat, cell_metadata = ref_cells$obs)

scdb_add_mat(atlas_nm, atlas)

### Merged matrix

Merges query and atlas into `query_atlas`, removes cells from neither input ("mystery" cells), and saves it under the same name.

In [ ]:
mat = scdb_mat(mat_nm)
atlas = scdb_mat(atlas_nm)

query_atlas = scm_merge_mats(mat, atlas)

scdb_add_mat(merged_nm, query_atlas)

In [ ]:
mat = scdb_mat(mat_nm)
atlas = scdb_mat(atlas_nm)

In [ ]:
query_atlas = scdb_mat(merged_nm)

In [ ]:
mystery_cells = colnames(query_atlas@mat)[!colnames(query_atlas@mat) %in% c(colnames(mat@mat), colnames(atlas@mat))]
cls_to_keep = colnames(query_atlas@mat)[!colnames(query_atlas@mat) %in% mystery_cells]

In [ ]:
sum(colnames(atlas@mat) %in% colnames(query_atlas@mat))
sum(colnames(mat@mat) %in% colnames(query_atlas@mat))

sum(!colnames(query_atlas@mat) %in% c(colnames(mat@mat), colnames(atlas@mat)))

In [ ]:
query_atlas_f = scm_ignore_cells(scmat = query_atlas, ig_cells = mystery_cells, reverse = FALSE)
scdb_add_mat(merged_nm, query_atlas_f)

In [ ]:
query_atlas = scdb_mat(merged_nm)

## Gene selection

Removes from the merged matrix (result: `query_atlas_f`):
- Genes not among the atlas model's feature genes
- Genes on chr6, chr8, chr11 (Dnmt-TKO and Tet-TKO trisomies)
- Dnmt, Tet and Uhrf1 genes; nine imprinted genes
- Genes absent from the atlas

Feature genes: `T_vm = 0.1`, `T_tot = 50`, `T_top3 = 3`; saved as gene set `query_atlas`.

Caveats:
- Dnmt-DKO trisomies (chr1, chr14) are not excluded.
- TSS intervals come from a separate misha root (`mm10_db_dir` in `paths.R`), not `mm10_trackdb`.

In [ ]:
query_atlas = scdb_mat(merged_nm)

In [ ]:
non_selected_genes_ref = rownames(ref_mcs$var)[!ref_mcs$var$selected_gene]

In [ ]:
library(misha)
gsetroot(mm10_db_dir)
tss = gintervals.load('intervs.global.tss')

trisomy_genes = tss %>% filter(chrom %in% c("chr6", "chr8", "chr11")) %>% pull(geneSymbol)

In [ ]:
mut_genes = c("Dnmt3a", "Dnmt3b", "Dnmt1", "Uhrf1", "Tet1", "Tet2", "Tet3")
imprinted_genes = c('Igf2', 'Dlk1', 'H19', 'Phlda2', 'Meg3', 'Mest', "Peg10", "Peg3", "Cdkn1c")

In [ ]:
bad_gns = unique(c(non_selected_genes_ref, trisomy_genes, mut_genes, imprinted_genes))

In [ ]:
grep("Rpl", non_selected_genes_ref, v=T)

In [ ]:
# Remove non-selected genes and trisomy genes from gset mat
mcell_mat_ignore_genes(new_mat_id = merged_nm_f, mat_id = merged_nm, ig_genes =  bad_gns, reverse = F)

In [ ]:
query_atlas_f = scdb_mat(merged_nm_f)
excluded_genes = rownames(query_atlas_f@mat)[!rownames(query_atlas_f@mat) %in% ref_mcs$var_names]

In [ ]:
mcell_mat_ignore_genes(new_mat_id = merged_nm_f, mat_id = merged_nm_f, ig_genes = c(bad_gns, excluded_genes), reverse = F)

In [ ]:
query_atlas_f = scdb_mat(merged_nm_f)

In [ ]:
T_vm = 0.1
T_tot=50
T_top3=3

In [ ]:
mcell_add_gene_stat(mat_id = merged_nm_f, gstat_id = merged_nm, force=T)

mcell_gset_filter_varmean(merged_nm, merged_nm, T_vm=T_vm, force_new=T)
mcell_gset_filter_cov(merged_nm, merged_nm, T_tot=T_tot, T_top3=T_top3)

## Balanced kNN graph

Inlined copy of metacell's graph construction on `query_atlas` (K = 100), saved as cell graph `query_atlas`.

In [ ]:
Knn = 100

In [ ]:
gset = scdb_gset(merged_nm)
nms = names(gset@gene_set)

In [ ]:
mat_id = merged_nm
gset_id = merged_nm
graph_id = merged_nm
K = Knn
dsamp = T

In [ ]:
downsample_n = NA
gene_names_src_targ = NULL

In [ ]:
gset = scdb_gset(gset_id)
if(is.null(gset)) {
    stop("MC-ERR non existing gset in gset_get_feat_mat, id ", gset_id)
}
mat = scdb_mat(mat_id)
if(is.null(mat)) {
    stop("MC-ERR non existing mat in gset_get_geat_mat, id ", mat_id)
}

if(!is.null(gene_names_src_targ)) {
    gnames_df = scdb_gene_names_xref()
    if(length(intersect(gene_names_src_targ, colnames(gnames_df)))!=2) {
        stop("Gene names conversion src/targ ", 
                        paste(gene_names_src_targ, collapse="/"), 
                                " are not in the scdb translation table")
    }
}

In [ ]:
if(is.na(downsample_n)) {
        downsample_n = scm_which_downsamp_n(mat)
    }
message("will downsample the matrix, N= ", downsample_n)
umis = scm_downsamp(mat@mat, downsample_n)

In [ ]:
csz = colSums(as.matrix(mat@mat))

In [ ]:
umis_cbind = cbind(umis, mat@mat[,names(csz < downsample_n)])

In [ ]:
umis_ordered = umis_cbind[,colnames(mat@mat)]

In [ ]:
if(ncol(umis_ordered) != ncol(mat@mat)) {
        stop("size mismatch after dsamp umis on gset")
    }

In [ ]:
nms = names(gset@gene_set)
nms = intersect(nms, rownames(umis_ordered))

if(length(nms) == 0) {
    stop("get feature matrix with zero overlap of gene names with gset_id ", gset_id, " mat ", mat_id)
}
feat_ds = umis_ordered[nms,]
if(!is.null(gene_names_src_targ)) {
    gnames_df = scdb_gene_names_xref()
    key_s = gene_names_src_targ[1]
    key_t = gene_names_src_targ[2]
    f = !is.na(gnames_df[,key_t]) & !duplicated(gnames_df[,key_t])
    new_gnames = gnames_df[f,key_t]
    names(new_gnames) = gnames_df[f, key_s]
    nms_targ = intersect(rownames(feat_ds),names(new_gnames))
    if(length(nms_targ) < 1) {
        stop("get feature matrix with zero overlap of gene names that can be translated given keys ", key_s, " to ", key_t)
    }

    feat_ds = feat_ds[nms_targ,]
    rownames(feat_ds) = new_gnames[rownames(feat_ds)]
}

In [ ]:
feat = feat_ds

In [ ]:
k_nonz_exp = tgconfig::get_param("scm_k_nonz_exp", "metacell")
feat = log2(1+k_nonz_exp*as.matrix(feat))

In [ ]:
message("will build balanced knn graph on ", ncol(feat), " cells and ", nrow(feat), " genes, this can be a bit heavy for >20,000 cells")
k_alpha = tgconfig::get_param("scm_balance_graph_k_alpha", "metacell")
k_beta = tgconfig::get_param("scm_balance_graph_k_beta", "metacell")
k_expand = tgconfig::get_param("scm_balance_graph_k_expand", "metacell")

In [ ]:
gr = tgs_cor_graph(x=feat, knn=K, k_expand=k_expand, k_alpha=k_alpha, k_beta=k_beta)

In [ ]:
cnames = colnames(feat)
colnames(gr) = c("mc1", "mc2", "w")

scdb_add_cgraph(graph_id, tgCellGraph(gr, cnames))

## Transcriptional time per embryo

- Drops query cells with > 1.5% UMIs on ExE ectoderm genes; groups the rest by embryo and KO/host.
- Each query cell gets the time of its strongest atlas neighbour; each atlas cell gets that of its strongest atlas neighbour from another time point. Atlas cells without a time are dropped.
- Correlates each query group's time distribution with each atlas time's, smooths along time (`spar = 0.8`), and takes the best time.
- Embryos with any group of ≤ 100 matched cells are left out of the table.

In [ ]:
cells_X = t(data.matrix(cells$X))

In [ ]:
ExE_genes = read.csv('./my_data/ExE/ExE_gene_names.csv', header = F)[,1]

In [ ]:
ExE_fractions = colSums(cells_X[ExE_genes,])/colSums(cells_X)
sum(ExE_fractions > 0.015)

In [ ]:
cls_to_keep = names(ExE_fractions)[ExE_fractions < 0.015]

In [ ]:
graph_id = merged_nm

In [ ]:
query_cls_md = deframe(cells$obs %>% rownames_to_column("cell") %>% mutate(embryo_KO_host = paste(embryo, KO_host, sep = "_")) %>% select(cell, embryo_KO_host))
query_cls_md = query_cls_md[cls_to_keep]

In [ ]:
atlas_time = deframe(ref_cells$obs %>% select(cell, developmental_time) %>% distinct()) 

In [ ]:
atlas_time[is.na(atlas_time)] = 0

In [ ]:
max(atlas_time)

In [ ]:
time_matches = get_query_and_atlas_time_dist(query_cls_md, atlas_time, graph_id)

In [ ]:
atlas_time_dist = time_matches[[1]]
query_time_dist = time_matches[[2]]
atlas_time_dist = atlas_time_dist[rownames(atlas_time_dist) != "0", colnames(atlas_time_dist) != "0"]
query_time_dist = query_time_dist[,colnames(query_time_dist) != "0"]

In [ ]:
query_ref_cor = tgstat::tgs_cor(t(as.matrix(as.data.frame.matrix(query_time_dist))),
                                t(as.matrix(as.data.frame.matrix(atlas_time_dist))))

In [ ]:
under_represented = names(rowSums(query_time_dist))[rowSums(query_time_dist) <= 100]
quantile(rowSums(query_time_dist), seq(0, 1, 0.05))["10%"]

In [ ]:
dev_time = as.character(sort(as.numeric(colnames(query_ref_cor))))

In [ ]:
best_fit = apply(query_ref_cor, 1, function(emb_cor) {
      
    smooth_spline = smooth.spline(x = dev_time, y = emb_cor, spar = 0.8)
    y = smooth_spline$y
      
    return(which.max(y))
      
  })

In [ ]:
best_time = dev_time[best_fit]
names(best_time) = names(best_fit)

best_time_df = enframe(best_time) %>% rename(embryo = 1, transcriptional_time = 2) %>% 
                                      mutate(transcriptional_time = as.numeric(transcriptional_time),
                                             less_than_100 = ifelse(embryo %in% under_represented, TRUE, FALSE),
                                             embryo = sapply(embryo, function(x) {
                                                                        x = sub("_control", "__control", x)
                                                                        x = sub("_KO", "__KO", x)
                                                                        x = sub("_host", "__host", x)})) %>% 
                                      separate(embryo, into = c("embryo", "KO_host"), sep = "__", extra = "merge", remove = TRUE)

In [ ]:
write.csv(best_time_df %>% filter(!embryo %in% (best_time_df %>% filter(less_than_100) %>% pull(embryo))), out_path("./my_data/ref/transcriptional_time_best_ref.csv"))

Writes the ED 2d table. The next cell only shows control embryos.

In [ ]:
best_time_df %>% filter(grepl("Ctrl", embryo))